# Deep Ensembles for a Parametric 1D Poisson Problem

This notebook uses **deep ensembles** as a regression-first uncertainty baseline
for a simple scientific surrogate. The governing equation is

$$
-u''(x)=f(x;a,b),\quad x\in[0,1],\qquad u(0)=u(1)=0,
$$

with forcing

$$
f(x;a,b)=a\sin(\pi x)+b\sin(3\pi x).
$$

The exact solution is known analytically, which makes it easy to evaluate both
fit error and uncertainty quality.

**Input to the model:** the coordinate $x$ and source parameters $(a,b)$.

**Output of the model:** the scalar solution value $u(x)$.

Primary paper:

- Lakshminarayanan, Pritzel, Blundell (2017), *Simple and Scalable Predictive Uncertainty Estimation using Deep Ensembles*.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / 'src').exists():
    repo_root = repo_root.parent.parent
sys.path.insert(0, str(repo_root / 'src'))

import math
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset

from deepuq.methods import DeepEnsembleWrapper
from deepuq.models import MLP

plt.rcParams['figure.figsize'] = (8, 4)
torch.manual_seed(7)
np.random.seed(7)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


## Dataset and physical meaning

This is a steady-state diffusion / elasticity style problem. The source term
$f(x;a,b)$ acts like a distributed load, and the solution $u(x)$ is the static
response. By varying $(a,b)$, we define a family of PDEs and learn the mapping
from parameters plus coordinate to the corresponding solution value.

The OOD split uses stronger and higher-frequency forcing parameters, so the
ensemble should show a wider uncertainty band there.


In [ ]:
config = {
    'n_train': 320,
    'n_val': 80,
    'n_test': 80,
    'n_ood': 80,
    'n_x': 128,
    'ensemble_size': 5,
    'epochs': 160,
    'batch_size': 256,
    'lr': 2e-3,
}


def exact_solution(x, a, b):
    return (a / math.pi**2) * torch.sin(math.pi * x) + (b / (9.0 * math.pi**2)) * torch.sin(3.0 * math.pi * x)


def sample_params(n, ood=False):
    if not ood:
        a = 0.5 + torch.rand(n, 1)
        b = -0.2 + 0.4 * torch.rand(n, 1)
    else:
        a = 1.5 + torch.rand(n, 1)
        b = 0.4 + 0.4 * torch.rand(n, 1)
    return a, b


def build_point_dataset(n_samples, *, ood=False):
    x = torch.linspace(0.0, 1.0, config['n_x']).unsqueeze(-1)
    a, b = sample_params(n_samples, ood=ood)
    x_grid = x.unsqueeze(0).repeat(n_samples, 1, 1)
    a_grid = a.unsqueeze(1).repeat(1, config['n_x'], 1)
    b_grid = b.unsqueeze(1).repeat(1, config['n_x'], 1)
    y = exact_solution(x_grid, a_grid, b_grid)
    features = torch.cat([x_grid, a_grid, b_grid], dim=-1)
    return features.reshape(-1, 3), y.reshape(-1, 1), x.squeeze(-1), (a, b)

train_x, train_y, x_axis, train_params = build_point_dataset(config['n_train'])
val_x, val_y, _, _ = build_point_dataset(config['n_val'])
test_x, test_y, _, test_params = build_point_dataset(config['n_test'])
ood_x, ood_y, _, ood_params = build_point_dataset(config['n_ood'], ood=True)

train_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=config['batch_size'], shuffle=True)
val_loader = DataLoader(TensorDataset(val_x, val_y), batch_size=config['batch_size'])


In [ ]:
# Visualize one in-domain and one OOD forcing/solution pair.
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, params, title in zip(axes, [test_params, ood_params], ['In-domain', 'OOD']):
    a, b = params[0][0].item(), params[1][0].item()
    forcing = a * torch.sin(math.pi * x_axis) + b * torch.sin(3.0 * math.pi * x_axis)
    solution = exact_solution(x_axis.unsqueeze(-1), torch.tensor([[a]]), torch.tensor([[b]])).squeeze(-1)
    ax.plot(x_axis, forcing, label='forcing $f(x)$')
    ax.plot(x_axis, solution, label='solution $u(x)$')
    ax.set_title(f'{title}: a={a:.2f}, b={b:.2f}')
    ax.set_xlabel('x')
axes[0].legend()
plt.tight_layout()
plt.show()


## Train the ensemble

Each member is an independent `MLP`. The ensemble mean is the prediction, and
the spread between members is the epistemic uncertainty estimate.


In [ ]:
models = [MLP(3, [128, 128, 128], 1, p_drop=0.0).to(device) for _ in range(config['ensemble_size'])]
ensemble = DeepEnsembleWrapper(models).to(device)
ensemble.fit(
    train_loader,
    epochs=config['epochs'],
    loss_fn=torch.nn.functional.mse_loss,
    lr=config['lr'],
    weight_decay=1e-4,
    device=device,
    seed=13,
)


In [ ]:
@torch.inference_mode()
def evaluate_dataset(x, y):
    mean, var = ensemble.predict(x.to(device))
    rmse = torch.mean((mean.cpu() - y) ** 2).sqrt().item()
    return mean.cpu(), var.cpu(), rmse

mean_test, var_test, rmse_test = evaluate_dataset(test_x, test_y)
mean_ood, var_ood, rmse_ood = evaluate_dataset(ood_x, ood_y)
print({'test_rmse': rmse_test, 'ood_rmse': rmse_ood})


In [ ]:
# Plot predictive bands for a representative in-domain and OOD parameter set.

def plot_band(features, targets, mean, var, title):
    n = config['n_x']
    idx = 0
    x = features[idx * n:(idx + 1) * n, 0]
    y_true = targets[idx * n:(idx + 1) * n, 0]
    y_mean = mean[idx * n:(idx + 1) * n, 0]
    y_std = var[idx * n:(idx + 1) * n, 0].sqrt()
    plt.figure(figsize=(8, 4))
    plt.plot(x, y_true, label='true solution', linewidth=2)
    plt.plot(x, y_mean, label='ensemble mean', linewidth=2)
    plt.fill_between(x.numpy(), (y_mean - 2 * y_std).numpy(), (y_mean + 2 * y_std).numpy(), alpha=0.3, label='95% band')
    plt.title(title)
    plt.xlabel('x')
    plt.ylabel('u(x)')
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_band(test_x, test_y, mean_test, var_test, f'In-domain prediction, RMSE={rmse_test:.4e}')
plot_band(ood_x, ood_y, mean_ood, var_ood, f'OOD prediction, RMSE={rmse_ood:.4e}')


In [ ]:
# Compare average uncertainty levels on in-domain and OOD sets.
plt.figure(figsize=(6, 4))
plt.bar(['test', 'ood'], [var_test.sqrt().mean().item(), var_ood.sqrt().mean().item()], color=['tab:blue', 'tab:orange'])
plt.ylabel('mean predictive std')
plt.title('Ensemble uncertainty grows on OOD forcing patterns')
plt.tight_layout()
plt.show()
